In [1]:
import os
import tensorflow as tf
import numpy as np
from flask import Flask, request, jsonify, render_template
from tensorflow.keras.preprocessing.image import img_to_array, load_img

# Load the trained model
model = tf.keras.models.load_model('leaf_detection_densenet_model.h5')

# Define class names and cures for diseases
class_names =['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 
               'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 
               'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 
               'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 
               'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 
               'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 
               'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tea__algal_spot', 'Tea__brown_blight', 'Tea__gray_blight', 
               'Tea__healthy', 'Tea__helopeltis', 'Tea__red_spot', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 
               'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 
               'Tomato___Target_Spot', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus', 'Tomato___healthy', 'train']
cures = {
    'Apple___Apple_scab': 'Use fungicides and remove infected leaves.',
    'Apple___Black_rot': 'Remove and destroy infected fruit and leaves.',
    'Apple___Cedar_apple_rust': 'Apply appropriate fungicides and prune affected areas.',
    'Apple___healthy': 'No action needed. The plant is healthy.',
    'Blueberry___healthy': 'No action needed. The plant is healthy.',
    'Cherry_(including_sour)___Powdery_mildew': 'Apply sulfur-based fungicides and ensure good air circulation.',
    'Cherry_(including_sour)___healthy': 'No action needed. The plant is healthy.',
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 'Rotate crops and use resistant varieties.',
    'Corn_(maize)___Common_rust_': 'Use resistant hybrids and apply fungicides if necessary.',
    'Corn_(maize)___Northern_Leaf_Blight': 'Use resistant varieties and apply fungicides.',
    'Corn_(maize)___healthy': 'No action needed. The plant is healthy.',
    'Grape___Black_rot': 'Apply fungicides and remove mummified berries.',
    'Grape___Esca_(Black_Measles)': 'Prune and remove infected wood.',
    'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)': 'Use fungicides and improve air circulation.',
    'Grape___healthy': 'No action needed. The plant is healthy.',
    'Orange___Haunglongbing_(Citrus_greening)': 'Remove infected trees and control psyllid populations.',
    'Peach___Bacterial_spot': 'Apply copper-based bactericides and remove infected leaves.',
    'Peach___healthy': 'No action needed. The plant is healthy.',
    'Pepper,_bell___Bacterial_spot': 'Use resistant varieties and apply copper sprays.',
    'Pepper,_bell___healthy': 'No action needed. The plant is healthy.',
    'Potato___Early_blight': 'Use fungicides and remove affected leaves.',
    'Potato___Late_blight': 'Apply fungicides and destroy infected plants.',
    'Potato___healthy': 'No action needed. The plant is healthy.',
    'Raspberry___healthy': 'No action needed. The plant is healthy.',
    'Soybean___healthy': 'No action needed. The plant is healthy.',
    'Squash___Powdery_mildew': 'Apply fungicides and ensure good air circulation.',
    'Strawberry___Leaf_scorch': 'Remove infected leaves and apply fungicides.',
    'Strawberry___healthy': 'No action needed. The plant is healthy.',
    'Tea__algal_spot': 'Proper pruning and removal of infected leaves. Use of appropriate fungicides if necessary.',
    'Tea__brown_blight': 'Prune and destroy affected leaves. Apply fungicides such as mancozeb or copper-based products.',
    'Tea__red_spot': 'Prune affected leaves and maintain good plant hygiene. Apply fungicides containing copper or mancozeb.',
    'Tea__helopeltis': 'Spray neem oil mixed with water and a small amount of soap directly on the affected tea plants.',
    'Tea__gray_blight': 'Improve air circulation and reduce humidity. Apply fungicides like chlorothalonil or mancozeb.',
    'Tea__healthy': 'No action needed for healthy leaves.',
    'Tomato___Bacterial_spot': 'Use copper-based sprays and remove infected plants.',
    'Tomato___Early_blight': 'Apply fungicides and practice crop rotation.',
    'Tomato___Late_blight': 'Apply fungicides and remove infected plants.',
    'Tomato___Leaf_Mold': 'Ensure good air circulation and apply fungicides.',
    'Tomato___Septoria_leaf_spot': 'Remove infected leaves and apply fungicides.',
    'Tomato___Spider_mites Two-spotted_spider_mite': 'Use miticides and encourage natural predators.',
    'Tomato___Target_Spot': 'Apply fungicides and practice crop rotation.',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus': 'Control whitefly population and use resistant varieties.',
    'Tomato___Tomato_mosaic_virus': 'Remove infected plants and sanitize tools.',
    'Tomato___healthy': 'No action needed. The plant is healthy.'
}

# Initialize the Flask app
app = Flask(__name__)

# Function to run inference on a single image and return the predicted label
def predict_image(image_path):
    img = load_img(image_path, target_size=(224, 224))
    img_array = img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.0

    prediction = model.predict(img_array)
    predicted_class = np.argmax(prediction, axis=1)
    
    predicted_label = class_names[predicted_class[0]]
    predicted_cure = cures.get(predicted_label, "No cure information available.")
    
    return predicted_label, predicted_cure

@app.route('/')
def welcome():
    return render_template('welcome.html')

@app.route('/main')
def main():
    return render_template('main.html')

# Route to handle image upload and prediction
@app.route('/predict', methods=['POST'])
def predict():
    if 'imagefile' not in request.files:
        return jsonify({"error": "No file part"}), 400
    file = request.files['imagefile']
    if file.filename == '':
        return jsonify({"error": "No selected file"}), 400
    if file:
        # Save the file to a temporary location
        file_path = os.path.join('uploads', file.filename)
        file.save(file_path)

        # Perform prediction
        predicted_label, predicted_cure = predict_image(file_path)

        # Clean up the saved file
        os.remove(file_path)

        # Return the prediction result as JSON
        return jsonify({"prediction": predicted_label, "cure": predicted_cure})

if __name__ == '__main__':
    if not os.path.exists('uploads'):
        os.makedirs('uploads')
    app.run(debug=False)
    
    #run in 3.11.7 version




 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [23/Aug/2024 22:04:30] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [23/Aug/2024 22:04:31] "GET /static/images/welcome.jpg HTTP/1.1" 200 -
127.0.0.1 - - [23/Aug/2024 22:04:31] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [23/Aug/2024 22:04:41] "GET /main HTTP/1.1" 200 -
127.0.0.1 - - [23/Aug/2024 22:04:41] "GET /static/images/welcome.jpg HTTP/1.1" 304 -
127.0.0.1 - - [23/Aug/2024 22:04:41] "GET /main HTTP/1.1" 200 -


1/1 [==============================] - 3s 3s/step


127.0.0.1 - - [23/Aug/2024 22:05:01] "POST /predict HTTP/1.1" 200 -


1/1 [==============================] - 0s 180ms/step


...

127.0.0.1 - - [23/Aug/2024 22:05:28] "POST /predict HTTP/1.1" 200 -


1/1 [==============================] - 0s 250ms/step


127.0.0.1 - - [23/Aug/2024 22:05:33] "POST /predict HTTP/1.1" 200 -


1/1 [==============================] - 0s 164ms/step


127.0.0.1 - - [23/Aug/2024 22:05:50] "POST /predict HTTP/1.1" 200 -
